# 02 - Nessie Branching (Git for data)
Branch the whole catalog off `main`, experiment in isolation, then **keep** the work (merge) or **revert** (drop the branch) - production `main` is never at risk.

Run notebook 01 first (it creates the table), or the setup cell below recreates it.

## 1. Start Spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("lakehouse-notebook")
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
            "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.77.1,"
            "software.amazon.awssdk:bundle:2.24.8,"
            "software.amazon.awssdk:url-connection-client:2.24.8")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
            "org.projectnessie.spark.extensions.NessieSparkSessionExtensions")
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1")
    .config("spark.sql.catalog.nessie.ref", "main")
    .config("spark.sql.catalog.nessie.authentication.type", "NONE")
    .config("spark.sql.catalog.nessie.warehouse", "s3://warehouse")
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true")
    .config("spark.sql.catalog.nessie.s3.access-key-id", "minioadmin")
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "minioadmin")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark ready:", spark.version)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12 added as a dependency
software.amazon.awssdk#bundle added as a dependency
software.amazon.awssdk#url-connection-client added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8651d89f-1bfe-4843-a268-7e93791f3df2;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
	found org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12;0.77.1 in central
	found software.amazon.awssdk#bundle;2.24.8 in central
	found software.amazon.awssdk#url-connection-client;2.24.8 in central
	found software.amazon.awssdk#utils;2.24.8 in central
	found org.reactivestreams#reactive-streams;1.0.4 in central
	found software.amazon.awssdk#annotations;2.24.8 in central
	found org.slf4j#slf4j-

Spark ready: 3.5.2


In [2]:
TABLE = "nessie.demo.card_txns"
BRANCH = "etl_experiment"
def count():
    return spark.sql(f"SELECT COUNT(*) c FROM {TABLE}").collect()[0]["c"]

## 2. Make sure a table exists on main

In [3]:
spark.sql("USE REFERENCE main IN nessie")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.demo")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE} (
        transaction_id STRING, card_id STRING, amount DOUBLE,
        currency STRING, txn_status STRING
    ) USING iceberg
""")
if count() == 0:
    spark.sql(f"""
        INSERT INTO {TABLE} VALUES
        ('TXN1','CARD001',120.50,'CHF','APPROVED'),
        ('TXN2','CARD002', 75.00,'EUR','APPROVED'),
        ('TXN3','CARD003',240.00,'CHF','APPROVED')
    """)
print("main row count =", count())

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


main row count = 3


## 3. Create a branch off main and switch to it

In [4]:
spark.sql(f"DROP BRANCH IF EXISTS {BRANCH} IN nessie")
spark.sql(f"CREATE BRANCH {BRANCH} IN nessie FROM main")
spark.sql(f"USE REFERENCE {BRANCH} IN nessie")
spark.sql("LIST REFERENCES IN nessie").show(truncate=False)

+-------+--------------+----------------------------------------------------------------+
|refType|name          |hash                                                            |
+-------+--------------+----------------------------------------------------------------+
|Branch |etl_experiment|0c65ea47d77ca1b471bd06687a578119d6e66d6ceaee7c6982c15d7ab3fbaf8a|
|Branch |main          |0c65ea47d77ca1b471bd06687a578119d6e66d6ceaee7c6982c15d7ab3fbaf8a|
+-------+--------------+----------------------------------------------------------------+



## 4. Risky edit - ONLY on the branch

In [5]:
spark.sql(f"DELETE FROM {TABLE} WHERE currency = 'EUR'")
spark.sql(f"INSERT INTO {TABLE} VALUES ('TXN9','CARD009',999.99,'CHF','APPROVED')")
print("branch row count =", count())
spark.sql(f"SELECT * FROM {TABLE} ORDER BY transaction_id").show()

branch row count = 3


+--------------+-------+------+--------+----------+
|transaction_id|card_id|amount|currency|txn_status|
+--------------+-------+------+--------+----------+
|          TXN1|CARD001| 120.5|     CHF|  APPROVED|
|          TXN3|CARD003| 240.0|     CHF|  APPROVED|
|          TXN9|CARD009|999.99|     CHF|  APPROVED|
+--------------+-------+------+--------+----------+



## 5. Switch back to main - it is untouched
## Query to see the branch changes in Dremio : SELECT * FROM lakehouse.demo.card_txns AT BRANCH "etl_experiment";

In [14]:
spark.sql("USE REFERENCE main IN nessie")
print("main row count =", count(), " (unchanged)")
spark.sql(f"SELECT * FROM {TABLE} ORDER BY transaction_id").show()

main row count = 3  (unchanged)
+--------------+-------+------+--------+----------+
|transaction_id|card_id|amount|currency|txn_status|
+--------------+-------+------+--------+----------+
|          TXN1|CARD001| 120.5|     CHF|  APPROVED|
|          TXN2|CARD002|  75.0|     EUR|  APPROVED|
|          TXN3|CARD003| 240.0|     CHF|  APPROVED|
+--------------+-------+------+--------+----------+



## 6. Decide: keep or revert
- **Keep:** run the MERGE line.
- **Revert:** drop the branch and the experiment vanishes.

In [15]:
# KEEP the work instead:
spark.sql(f"MERGE BRANCH {BRANCH} INTO main IN nessie")

# REVERT / discard:
#spark.sql(f"DROP BRANCH IF EXISTS {BRANCH} IN nessie")
#spark.sql("LIST REFERENCES IN nessie").show(truncate=False)
#print("main row count still =", count())

DataFrame[name: string, hash: string]

In [16]:
spark.sql("USE REFERENCE main IN nessie")
print("main row count =", count(), " (unchanged)")
spark.sql(f"SELECT * FROM {TABLE} ORDER BY transaction_id").show()

main row count = 3  (unchanged)
+--------------+-------+------+--------+----------+
|transaction_id|card_id|amount|currency|txn_status|
+--------------+-------+------+--------+----------+
|          TXN1|CARD001| 120.5|     CHF|  APPROVED|
|          TXN3|CARD003| 240.0|     CHF|  APPROVED|
|          TXN9|CARD009|999.99|     CHF|  APPROVED|
+--------------+-------+------+--------+----------+

